# Automating a Ghana Agrivoltaics Data Pipeline

This notebook is the code-along workspace for the 3-hour tutorial.

**Main question:** Can Ghanaian farms combine solar power generation with crop production without reducing harvest performance?

We will use the FAIR Forward Ghana agrivoltaics dataset and build the pipeline in stages:

1. Load raw files
2. Inspect and profile data
3. Define a data contract
4. Clean and standardize fields
5. Validate before trusting
6. Transform into crop insights
7. Save outputs and interpret responsibly


## Dataset

- FAIR Forward catalog: https://fair-forward.github.io/datasets/
- Kaggle dataset: https://www.kaggle.com/datasets/responsibleailab/agrivoltaic-dataset-ghana
- Country: Ghana
- Provider: Responsible AI Lab, KNUST
- License: CC-BY 4.0
- Crops: tomatoes, chilli pepper, and eggplant

Place downloaded files in `data/raw/` before running the notebook.


## Module 1: Load Raw Files

**Big idea:** The first job of a pipeline is to reliably find and load its inputs.


In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORTS_DIR = ROOT / "outputs" / "reports"
CHARTS_DIR = ROOT / "outputs" / "charts"

for directory in [RAW_DIR, PROCESSED_DIR, REPORTS_DIR, CHARTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Raw data directory:", RAW_DIR)


In [ ]:
raw_files = sorted(RAW_DIR.glob("*"))

for path in raw_files:
    print(path.name)

if not raw_files:
    print("No files found. Download and extract the Kaggle dataset into data/raw/.")


In [ ]:
tabular_files = [
    path for path in raw_files
    if path.suffix.lower() in [".csv", ".xlsx", ".xls"]
]

tables = {}

for file in tabular_files:
    if file.suffix.lower() == ".csv":
        tables[file.stem] = pd.read_csv(file)
    else:
        tables[file.stem] = pd.read_excel(file)

for name, df in tables.items():
    print(name, df.shape)


### Checkpoint

You should have a `tables` dictionary where each key is a table name and each value is a DataFrame.


## Module 2: Inspect And Profile

**Big idea:** Inspection is where you earn the right to transform. Do not trust the file just because it opened.


In [ ]:
for name, df in tables.items():
    print(f"\n{name}")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    display(df.head())


In [ ]:
def profile_table(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[column].dtype) for column in df.columns],
        "missing_count": [int(df[column].isna().sum()) for column in df.columns],
        "missing_pct": [round(float(df[column].isna().mean() * 100), 2) for column in df.columns],
        "unique_values": [int(df[column].nunique(dropna=True)) for column in df.columns],
    })


profiles = {name: profile_table(df) for name, df in tables.items()}

for name, profile in profiles.items():
    profile.to_csv(REPORTS_DIR / f"profile_{name}.csv", index=False)
    print(f"\n{name}")
    display(profile)


### Your Turn

Identify:

- the likely crop performance table
- the likely energy table
- the crop column
- the treatment or plot column
- the yield or measurement column


## Module 3: Define The Data Contract

**Big idea:** A data contract gives the pipeline stable internal names even when raw files use messy source names.


In [ ]:
for name, df in tables.items():
    print(f"\n{name}")
    for column in df.columns:
        print("  ", column)


In [ ]:
# TODO: update these values after inspecting the real dataset columns.
crop_table_name = "replace_with_actual_table_name"

COLUMN_MAP = {
    "plot": "replace_with_raw_column_name",
    "treatment": "replace_with_raw_column_name",
    "crop": "replace_with_raw_column_name",
    "replicate": "replace_with_raw_column_name",
    "date": "replace_with_raw_column_name",
    "yield_value": "replace_with_raw_column_name",
    "yield_unit": "replace_with_raw_column_name",
}


In [ ]:
def clean_column_name(value):
    return (
        str(value)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def rename_with_contract(df: pd.DataFrame, column_map: dict) -> pd.DataFrame:
    rename_map = {
        raw_column: standard_column
        for standard_column, raw_column in column_map.items()
        if raw_column in df.columns
    }

    renamed = df.rename(columns=rename_map).copy()
    renamed.columns = [clean_column_name(column) for column in renamed.columns]
    return renamed


In [ ]:
# Run this cell after updating crop_table_name and COLUMN_MAP.
if crop_table_name in tables:
    crop_raw = tables[crop_table_name]
    crop_renamed = rename_with_contract(crop_raw, COLUMN_MAP)
    print(list(crop_renamed.columns))
    display(crop_renamed.head())
else:
    print("Update crop_table_name. Available tables:", list(tables.keys()))


## Module 4: Clean And Standardize

**Big idea:** Computers are fast, not wise. If labels are inconsistent, your summary will be inconsistent too.


In [ ]:
def clean_text(value):
    if pd.isna(value):
        return value
    return str(value).strip().lower().replace("_", " ")


def standardize_crop(value):
    value = clean_text(value)
    crop_map = {
        "tomato": "tomato",
        "tomatoes": "tomato",
        "chilli": "chilli pepper",
        "chili": "chilli pepper",
        "chilli pepper": "chilli pepper",
        "chili pepper": "chilli pepper",
        "egg plant": "eggplant",
        "eggplant": "eggplant",
    }
    return crop_map.get(value, value)


def standardize_treatment(value):
    value = clean_text(value)

    if value in {"control", "open sun", "open-sun", "no pv", "no panels"}:
        return "open_sun_control"

    if value in {"agrivoltaic", "raised pv", "under pv", "under panels", "solar pv"}:
        return "agrivoltaic"

    if value in {"ground mounted pv", "ground-mounted pv", "bare land pv"}:
        return "ground_mounted_pv"

    return value


In [ ]:
if "crop_renamed" in globals():
    crop_clean = crop_renamed.copy()

    if "crop" in crop_clean.columns:
        crop_clean["crop"] = crop_clean["crop"].apply(standardize_crop)

    if "treatment" in crop_clean.columns:
        crop_clean["treatment"] = crop_clean["treatment"].apply(standardize_treatment)

    if "date" in crop_clean.columns:
        crop_clean["date"] = pd.to_datetime(crop_clean["date"], errors="coerce")

    if "yield_value" in crop_clean.columns:
        crop_clean["yield_value"] = pd.to_numeric(crop_clean["yield_value"], errors="coerce")

    display(crop_clean.head())
else:
    print("Create crop_renamed first by completing the data contract step.")


In [ ]:
if "crop_clean" in globals():
    for column in ["crop", "treatment"]:
        if column in crop_clean.columns:
            print(column, sorted(crop_clean[column].dropna().unique()))


## Module 5: Validate Before Trusting

**Big idea:** Validation is where bad assumptions go to be noticed.


In [ ]:
ALLOWED_CROPS = {"tomato", "chilli pepper", "eggplant"}
ALLOWED_TREATMENTS = {"open_sun_control", "agrivoltaic", "ground_mounted_pv"}

errors = []

if "crop_clean" in globals():
    required_columns = ["crop", "treatment", "yield_value"]
    missing = [column for column in required_columns if column not in crop_clean.columns]
    if missing:
        errors.append(f"Missing required columns: {missing}")

    if "crop" in crop_clean.columns:
        invalid_crops = sorted(set(crop_clean["crop"].dropna()) - ALLOWED_CROPS)
        if invalid_crops:
            errors.append(f"Unexpected crop values: {invalid_crops}")

    if "treatment" in crop_clean.columns:
        invalid_treatments = sorted(set(crop_clean["treatment"].dropna()) - ALLOWED_TREATMENTS)
        if invalid_treatments:
            errors.append(f"Unexpected treatment values: {invalid_treatments}")

    if "yield_value" in crop_clean.columns and (crop_clean["yield_value"] < 0).any():
        errors.append("yield_value contains negative values")
else:
    errors.append("crop_clean has not been created")

validation_report = pd.DataFrame({
    "status": ["error"] * len(errors) if errors else ["ok"],
    "message": errors if errors else ["Validation passed"],
})

validation_report.to_csv(REPORTS_DIR / "validation_report.csv", index=False)
validation_report


### Your Turn

Add one more validation rule:

- duplicate check
- missing yield check
- invalid date check
- unexpected plot check


## Module 6: Transform Into Crop Insights

**Big idea:** Transformation turns cleaned records into decision-ready tables.


In [ ]:
if errors:
    print("Fix validation errors before trusting this summary:")
    for error in errors:
        print("-", error)
else:
    crop_summary = (
        crop_clean
        .dropna(subset=["crop", "treatment", "yield_value"])
        .groupby(["crop", "treatment"], dropna=False)
        .agg(
            observations=("yield_value", "count"),
            mean_yield=("yield_value", "mean"),
            median_yield=("yield_value", "median"),
            min_yield=("yield_value", "min"),
            max_yield=("yield_value", "max"),
        )
        .reset_index()
    )
    display(crop_summary)


In [ ]:
if "crop_summary" in globals():
    treatment_comparison = (
        crop_summary
        .pivot(index="crop", columns="treatment", values="mean_yield")
        .reset_index()
    )

    if {"agrivoltaic", "open_sun_control"}.issubset(treatment_comparison.columns):
        treatment_comparison["yield_difference"] = (
            treatment_comparison["agrivoltaic"] - treatment_comparison["open_sun_control"]
        )
        treatment_comparison["yield_difference_pct"] = (
            treatment_comparison["yield_difference"] / treatment_comparison["open_sun_control"] * 100
        ).round(2)

    display(treatment_comparison)


In [ ]:
if "crop_summary" in globals():
    crop_summary.to_csv(PROCESSED_DIR / "crop_performance_summary.csv", index=False)

if "treatment_comparison" in globals():
    treatment_comparison.to_csv(PROCESSED_DIR / "treatment_comparison.csv", index=False)

print("Processed outputs written to", PROCESSED_DIR)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if "crop_summary" in globals():
    plt.figure(figsize=(10, 6))
    sns.barplot(data=crop_summary, x="crop", y="mean_yield", hue="treatment")
    plt.title("Mean Crop Yield by Treatment")
    plt.xlabel("Crop")
    plt.ylabel("Mean yield")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "mean_crop_yield_by_treatment.png", dpi=150)
    plt.show()


## Wrap-Up: Responsible Interpretation

Write 5-7 sentences answering:

1. What does the processed data suggest?
2. Which crop appears most promising under agrivoltaic conditions?
3. What limitations should be communicated?
4. What extra data would strengthen the analysis?
5. What should we avoid claiming?

Remember: this is a pilot dataset. A clean pipeline supports better discussion, but it does not make the dataset universal.
